# 🔧 VibSense — Motor Fault Detection Model v3

**12 đặc trưng tính trên node từ cửa sổ 256 mẫu @ 800Hz**

| Feature | Mô tả |
|---------|-------|
| `x_rms`, `y_rms`, `z_rms` | RMS gia tốc từng trục (g) |
| `x_peak`, `y_peak`, `z_peak` | Giá trị đỉnh từng trục (g) |
| `x_cf`, `y_cf`, `z_cf` | Crest Factor từng trục |
| `vrms` | RMS tổng hợp 3 trục |
| `kurtosis` | Độ nhọn — nhạy với xung vòng bi |
| `skewness` | Độ lệch phân phối |

**Các trạng thái phân loại:**
- `normal` — Bình thường ✅
- `unbalance` — Mất cân bằng 🟡
- `misalignment` — Lệch trục 🟠
- `looseness` — Lỏng lẻo cơ khí 🔵
- `bearing_fault` — Hư vòng bi 🔴


## 1. Cài thư viện

In [ ]:
!pip install -q scikit-learn numpy pandas matplotlib seaborn

## 2. Import thư viện

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

np.random.seed(42)
print('✅ Import xong!')

## 3. Đọc dữ liệu từ file CSV

> Đặt các file CSV cùng thư mục với notebook, đặt tên theo nhãn trạng thái.
>
> Ví dụ: `normal.csv`, `bearing_fault.csv`, `unbalance.csv`, `misalignment.csv`, `looseness.csv`
>
> Mỗi file phải có các cột: **timestamp, x, y, z, vrms**

In [ ]:
import glob, os

# ─── Nhãn hợp lệ (khớp tên file CSV) ───────────────────────
VALID_LABELS = {'normal', 'unbalance', 'misalignment', 'looseness', 'bearing_fault'}

# ─── 12 đặc trưng từ node mới ───────────────────────────────
FEATURE_COLS = [
    'x_rms', 'y_rms', 'z_rms',
    'x_peak', 'y_peak', 'z_peak',
    'x_cf', 'y_cf', 'z_cf',
    'vrms', 'kurtosis', 'skewness'
]

# ─── Tìm tất cả file CSV cùng thư mục notebook ───────────────
csv_files = glob.glob('*.csv')
print(f'Tìm thấy {len(csv_files)} file CSV: {csv_files}')

frames  = []
skipped = []

for path in csv_files:
    label = os.path.splitext(os.path.basename(path))[0]
    if label not in VALID_LABELS:
        skipped.append(path)
        continue
    try:
        tmp = pd.read_csv(path)
        missing = [c for c in FEATURE_COLS if c not in tmp.columns]
        if missing:
            print(f'  ⚠️  {path}: thiếu cột {missing} — bỏ qua')
            skipped.append(path)
            continue
        tmp['label'] = label
        frames.append(tmp[FEATURE_COLS + ['label']])
        print(f'  ✅ {path}: {len(tmp)} mẫu — nhãn = {label}')
    except Exception as e:
        print(f'  ❌ {path}: lỗi đọc file — {e}')
        skipped.append(path)

if skipped:
    print(f'\n⚠️  Bỏ qua {len(skipped)} file: {skipped}')

if not frames:
    raise ValueError('Không tìm thấy file CSV hợp lệ!')

df = pd.concat(frames, ignore_index=True)

before = len(df)
df.dropna(subset=FEATURE_COLS, inplace=True)
if before - len(df) > 0:
    print(f'\n⚠️  Đã bỏ {before - len(df)} dòng NaN')

print(f'\n📊 Tổng mẫu: {len(df)}')
print(f'Phân bố lớp:\n{df["label"].value_counts()}')
df.groupby('label')[FEATURE_COLS].mean().round(3)


## 4. Trực quan hóa phân bố dữ liệu

In [ ]:
# Vẽ phân bố 4 đặc trưng đại diện
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
plot_features = ['vrms', 'kurtosis', 'skewness', 'x_peak']
titles = ['Vrms (RMS tổng hợp)', 'Kurtosis (độ nhọn)', 'Skewness (độ lệch)', 'X Peak (đỉnh trục X)']
colors = ['#2196F3','#4CAF50','#FF9800','#F44336','#9C27B0']

for ax, feat, title in zip(axes.flat, plot_features, titles):
    for (lbl, grp), c in zip(df.groupby('label'), colors):
        grp[feat].hist(ax=ax, bins=40, alpha=0.55, color=c, label=lbl, density=True)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Giá trị')
    ax.set_ylabel('Mật độ')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Phân bố đặc trưng theo trạng thái động cơ', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 5. Chia tập train / test & Huấn luyện Random Forest

In [ ]:
# FEATURE_COLS đã được định nghĩa ở Cell 3, không định nghĩa lại ở đây
# để tránh ghi đè thành 4 cột cũ

X_arr = df[FEATURE_COLS].to_numpy(dtype=float)
y_arr = df['label'].to_numpy(dtype=str)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_arr, y_arr,
    test_size=0.2, random_state=42, stratify=y_arr
)

print(f'Train: {len(X_tr)} mẫu | Test: {len(X_te)} mẫu')
print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')

model = RandomForestClassifier(
    n_estimators     = 300,
    max_depth        = 12,
    min_samples_leaf = 2,
    random_state     = 42,
    n_jobs           = -1
)
model.fit(X_tr, y_tr)
print('✅ Huấn luyện xong!')


## 6. Đánh giá mô hình

In [ ]:
y_pred = model.predict(X_te)
acc    = accuracy_score(y_te, y_pred)

print(f'\n🎯 Accuracy: {acc*100:.2f}%')
print('='*55)
print(classification_report(y_te, y_pred, digits=3))

In [ ]:
# Cross-validation 5-fold
cv_scores = cross_val_score(model, df[FEATURE_COLS].to_numpy(dtype=float), df['label'].to_numpy(dtype=str),
                             cv=5, scoring='accuracy', n_jobs=-1)
print(f'Cross-validation (5-fold):')
print(f'  Mỗi fold: {[f"{s:.3f}" for s in cv_scores]}')
print(f'  Trung bình: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

## 7. Ma trận nhầm lẫn (Confusion Matrix)

In [ ]:
LABEL_VI = {
    'normal':       'Bình thường',
    'unbalance':    'Mất cân bằng',
    'misalignment': 'Lệch trục',
    'looseness':    'Lỏng lẻo CK',
    'bearing_fault':'Hư vòng bi',
}
class_labels_vi = [LABEL_VI[c] for c in model.classes_]

cm = confusion_matrix(y_te, y_pred, labels=model.classes_)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_labels_vi,
    yticklabels=class_labels_vi,
    ax=ax, linewidths=0.5
)
ax.set_xlabel('Dự đoán', fontsize=12)
ax.set_ylabel('Thực tế',  fontsize=12)
ax.set_title('Ma trận nhầm lẫn — Random Forest', fontsize=13, fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 8. Tầm quan trọng của từng feature

In [ ]:
imp = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 5))
colors_bar = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(FEATURE_COLS)))
bars = plt.bar(imp['feature'], imp['importance'], color=colors_bar, edgecolor='white')
for bar, val in zip(bars, imp['importance']):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.title('Feature Importance — 12 đặc trưng', fontsize=12, fontweight='bold')
plt.ylabel('Importance score')
plt.xticks(rotation=30, ha='right')
plt.ylim(0, imp['importance'].max()*1.15)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(imp.to_string(index=False))


## 9. Test thử với dữ liệu mẫu từ từng file CSV

> Lấy 1 dòng ngẫu nhiên từ mỗi nhãn để kiểm tra model dự đoán đúng không

In [ ]:
LABEL_VI = {
    'normal':        'Bình thường ✅',
    'unbalance':     'Mất cân bằng 🟡',
    'misalignment':  'Lệch trục 🟠',
    'looseness':     'Lỏng lẻo CK 🔵',
    'bearing_fault': 'Hư vòng bi 🔴',
}

print(f'{"Nhãn thực tế":<45} {"Dự đoán":<22} {"Tin cậy"}')
print('-' * 80)

for label in sorted(df['label'].unique()):
    row = df[df['label'] == label].sample(1, random_state=42)
    arr   = row[FEATURE_COLS].to_numpy(dtype=float)
    pred  = model.predict(arr)[0]
    proba = model.predict_proba(arr)[0]
    conf  = max(proba)
    pred_vi = LABEL_VI.get(pred, pred)
    ok = '✅' if pred == label else '❌'
    print(f'{ok} Thực tế: {label:<35} → {pred_vi:<22} {conf*100:.1f}%')


## 10. Lưu model

> **Quan trọng:** Sau khi lưu, chép file `motor_fault_random_forest.pkl` vào thư mục cùng với `server.py` của VibSense Cloud.

In [ ]:
MODEL_PATH = 'motor_fault_random_forest.pkl'

with open(MODEL_PATH, 'wb') as f:
    pickle.dump(model, f)

# Verify: load lại và test bằng 1 dòng ngẫu nhiên từ df
with open(MODEL_PATH, 'rb') as f:
    loaded_model = pickle.load(f)

sample = df[FEATURE_COLS].iloc[0:1].to_numpy(dtype=float)
sample_label = df['label'].iloc[0]

print(f'✅ Model đã lưu: {MODEL_PATH}')
print(f'   Kiểm tra load lại: {loaded_model.predict(sample)[0]} (thực tế: {sample_label})')
print(f'   Classes: {list(loaded_model.classes_)}')
print(f'   Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')
print()
print('📋 Bước tiếp theo:')
print('   1. Chép file motor_fault_random_forest.pkl')
print('   2. Dán vào thư mục cùng với server.py')
print('   3. Chạy: python server.py')
print('   4. Mở http://localhost:8001')


---

## 📌 Tóm tắt pipeline tích hợp

```
Node ESP32 (IMU)
  │  Đo x, y, z (g) + tính vrms (mm/s)
  ▼
Gateway ESP32 (CSV Mode)
  │  Lưu buffer RAM → tải CSV về máy
  │  Đổi tên file theo nhãn: normal.csv, bearing_fault.csv ...
  ▼
Notebook (motor_fault_model_v2.ipynb)
  │  Đặt các file CSV cùng thư mục notebook
  │  Chạy lại từ Cell 3 để retrain
  ▼
VibSense Cloud (server.py)
  │  features = [x, y, z, vrms]
  │  prediction = model.predict(features)
  │  Lưu vào SQLite
  ▼
Dashboard (index.html)
  │  Hiển thị real-time mỗi 5 giây
```

### Quy trình retrain khi có dữ liệu mới
1. Thu thập CSV mới từ gateway cho từng trạng thái
2. Đổi tên file theo nhãn: `normal.csv`, `bearing_fault.csv`, ...
3. Đặt vào cùng thư mục notebook (ghi đè file cũ nếu muốn)
4. Chạy lại notebook từ **Cell 3** trở đi
5. Chép `motor_fault_random_forest.pkl` vào thư mục `server.py`
6. Gọi `POST /api/reload_model` hoặc restart server
